In [ ]:
import kagglehub
import pandas as pd
import os
from torch.utils.data import Dataset, DataLoader, random_split
from torch import nn, optim
from torch.optim import AdamW
import torch
from tqdm import tqdm
from transformers import DistilBertTokenizer, DistilBertForSequenceClassification, get_scheduler
from huggingface_hub import login
from datasets import Dataset

EMBED_DIM = 128
HIDDEN_DIM = 64
MAX_EPOCHS = 15

login(token=os.getenv("HF_TOKEN"))

path = kagglehub.dataset_download("snap/amazon-fine-food-reviews")
df = pd.read_csv(os.path.join(path, "Reviews.csv"), usecols=["Text", "Score"])

d:\LLM\trust-behaviours\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
tokenizer = DistilBertTokenizer.from_pretrained('distilbert-base-uncased')

def tokenize_batch(batch):
    return tokenizer(batch['Text'], padding="max_length", truncation=True, max_length=256)

In [3]:
df["Text"] = df["Text"].str.replace(r"<[^>]+>", " ", regex=True)
df["Text"] = df["Text"].str.replace(r"[^\w\s]", " ", regex=True)
df["labels"] = df["Score"] - 1

In [4]:
# Panda and Hugging Face stuff 
ds = Dataset.from_pandas(df[["Text", "labels"]])
ds = ds.map(tokenize_batch, batched=True)
ds.remove_columns(["Text"])
ds.set_format(type='torch', columns=['input_ids', 'attention_mask', 'labels'])

Map: 100%|██████████| 568454/568454 [01:21<00:00, 6963.20 examples/s]


In [5]:
train_size = int(0.8 * len(ds))
val_size = len(ds) - train_size
ds_train, ds_val = random_split(ds, [train_size, val_size])

dataloader_train = DataLoader(ds_train, batch_size=32, shuffle=True)
dataloader_val = DataLoader(ds_val, batch_size=32)

In [6]:
device = torch.device("cpu")
if torch.cuda.is_available():
  device = torch.device("cuda")
elif torch.backends.mps.is_available():
  device = torch.device("mps")

print(device)
print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else "No CUDA device")

model = DistilBertForSequenceClassification.from_pretrained("distilbert-base-uncased", num_labels=3).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.AdamW(model.parameters(), lr=2e-5)
scheduler = get_scheduler("linear", optimizer=optimizer, num_warmup_steps=0, num_training_steps=MAX_EPOCHS*len(dataloader_train))

cuda
NVIDIA GeForce RTX 4060 Ti


Loading weights: 100%|██████████| 100/100 [00:00<00:00, 6680.74it/s]
DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
pre_classifier.bias     | MISSING    | 
pre_classifier.weight   | MISSING    | 
classifier.weight       | MISSING    | 
classifier.bias         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [7]:
previous_val_loss = 1
for epoch in range(MAX_EPOCHS):
    # Entrainement
    model.train()
    for batch in tqdm(dataloader_train, desc=f"Epoch {epoch+1} - Training"):
        X_batch = batch['input_ids'].to(device)
        mask_batch = batch['attention_mask'].to(device)
        Y_batch = batch['labels'].to(device)

        optimizer.zero_grad()
        output = model(input_ids=X_batch, labels=Y_batch, attention_mask=mask_batch)
        output.loss.backward()
        optimizer.step()
        scheduler.step()

    # Validation
    model.eval()
    correct = 0
    total = 0
    val_loss = 0
    with torch.no_grad():
        for batch in tqdm(dataloader_val, desc=f"Epoch {epoch+1} - Validation"):
            X_batch = batch['input_ids'].to(device)
            mask_batch = batch['attention_mask'].to(device)
            Y_batch = batch['labels'].to(device)
            output = model(input_ids=X_batch, labels=Y_batch, attention_mask=mask_batch)
            predicted_value = output.logits.argmax(dim=1)
            correct += (Y_batch == predicted_value).sum().item()
            total += Y_batch.size(0)
            val_loss += output.loss.item()
    val_accuracy = correct / total
    val_loss /= len(dataloader_val)

    print(f"Epoch {epoch+1}/{MAX_EPOCHS} | val_loss: {val_loss:.4f} | val_accuracy: {val_accuracy:.4f} | lr: {optimizer.param_groups[0]['lr']:.6f}")
#    if (previous_val_loss - val_loss) < 0.005:
#        break
    previous_val_loss = val_loss

Epoch 1 - Training:   0%|          | 0/14212 [00:00<?, ?it/s]


AcceleratorError: CUDA error: device-side assert triggered
Search for `cudaErrorAssert' in https://docs.nvidia.com/cuda/cuda-runtime-api/group__CUDART__TYPES.html for more information.
Compile with `TORCH_USE_CUDA_DSA` to enable device-side assertions.


In [ ]:
torch.save(model.state_dict(), "../model.pt")